In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import tensorflow as tf
from tensorflow import keras

from src.dataset import (
    create_datasets,
    compute_class_weights,
    get_data_augmentation,
    get_class_names
)

from src.model import build_model

In [ ]:
train_ds, val_ds, test_ds = create_datasets()

augmentation = get_data_augmentation()

weights = compute_class_weights(train_ds)

model = build_model(augmentation)

Found 28709 files belonging to 7 classes.
Using 22968 files for training.
Found 28709 files belonging to 7 classes.
Using 5741 files for validation.
Found 7178 files belonging to 7 classes.


In [7]:
model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=0.001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)

In [8]:
checkpoint = keras.callbacks.ModelCheckpoint(
    filepath="../models/best_model.keras",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    class_weight=weights,
    callbacks=[
        checkpoint,
        early_stopping,
        reduce_lr
    ]
)